In [ ]:
import sys

from hydra.utils import instantiate
from einops import rearrange
from omegaconf import OmegaConf
import torch
import os
import sys

# Register custom resolver to handle multiplication in OmegaConf interpolation
OmegaConf.register_new_resolver("mul", lambda x, y: float(x) * float(y))
OmegaConf.register_new_resolver("div", lambda a, b: float(a) / float(b))

In [ ]:
snapshot_dir = "/svl/u/ravenh/lacwm/robot_world_models-raven-lam/projects/latent_action_models/data/experiments_0908/libero_sim_scratch_action_9/2025-11-24/19-41-14/"
config_path = f"{snapshot_dir}/.hydra"
print(config_path)

cfg = OmegaConf.load(config_path + "/config.yaml")

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("/svl/u/ravenh/lacwm/robot_world_models-raven-lam/projects/latent_action_models"))
os.environ['COSMOS_HOME'] = '/svl/u/ravenh/lacwm/Cosmos'

model = instantiate(cfg.model)
model = model.cuda().eval()
snapshot = torch.load(f"{snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
model.load_state_dict(snapshot["model"])
val_dataloader = instantiate(cfg.val_data_loader)

In [ ]:
libero_dataloader = val_dataloader[0]

In [ ]:
libero_dataset = libero_dataloader.dataset

In [53]:
libero_zs = []
rgbs = []

for i, libero_batch in enumerate(libero_dataset):
    rgb = libero_batch["rgb"][None].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    
    z = model._forward_inverse_model(x, libero_batch["actions"][None].cuda(), 
                                   morphology_index = libero_batch["morphology_index"][None].cuda(),
                                   ee_action_dim = libero_batch["ee_action_dim"][None].cuda() ).detach().cpu().numpy()
    libero_zs.append(z)
    rgbs.append(rgb.cpu().numpy())
    if i > 20:
        break

In [54]:
# run pca on the embeddinga and find the center of the cluster
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

def pca_cluster_representatives(
    embeddings, 
    pca_dim=32, 
    n_clusters=5,
    return_indices=False,
):
    """
    Args:
        embeddings: np.ndarray or torch.Tensor of shape (N, D)
        pca_dim: output PCA dimension
        n_clusters: number of clusters
        return_indices: if True, also return original indices of representatives

    Returns:
        reps: (n_clusters, D) representative embeddings close to cluster centroids
        indices (optional): list of original indices
    """

    # Convert torch → numpy if needed
    if "torch" in str(type(embeddings)):
        embeddings_np = embeddings.detach().cpu().numpy()
    else:
        embeddings_np = embeddings

    N, D = embeddings_np.shape

    # ---- 1. PCA Dim Reduction ----
    pca = PCA(n_components=pca_dim)
    reduced = pca.fit_transform(embeddings_np)   # shape (N, pca_dim)

    # ---- 2. Run K-means ----
    kmeans = KMeans(n_clusters=n_clusters, n_init="auto")
    labels = kmeans.fit_predict(reduced)

    reps = []
    rep_indices = []

    # ---- 3. For each cluster: find embedding closest to cluster mean ----
    for c in range(n_clusters):
        cluster_idx = np.where(labels == c)[0]

        if len(cluster_idx) == 0:
            continue  # empty cluster, unlikely but possible

        cluster_embeddings = reduced[cluster_idx]
        centroid = cluster_embeddings.mean(axis=0)

        # Euclidean distance to centroid
        distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)

        # index (within cluster) of representative embedding
        best_local_index = np.argmin(distances)
        best_global_index = cluster_idx[best_local_index]

        reps.append(embeddings_np[best_global_index])
        rep_indices.append(best_global_index)

    reps = np.stack(reps, axis=0)

    if return_indices:
        return reps, rep_indices, kmeans, labels, reduced
    return reps, kmeans, labels, reduced


In [59]:
libero_embedding = np.array(libero_zs)
libero_embedding = libero_embedding.reshape(-1, libero_embedding.shape[-1])


In [60]:
all_rgbs = np.array(rgbs)


In [61]:
all_rgbs = all_rgbs.reshape(-1, all_rgbs.shape[-3], all_rgbs.shape[-2], all_rgbs.shape[-1]) # N, C, H, W
all_rgbs = all_rgbs.transpose(0, 2, 3, 1) # N, H, W, C


In [62]:
n_clusters = 4

In [63]:
reps,rep_indices, kmeans, labels, reduced = pca_cluster_representatives(libero_embedding, n_clusters=n_clusters, return_indices=True)

In [ ]:
# --- Plot ---
import matplotlib.pyplot as plt
figsize=(6, 6)
plt.figure(figsize=figsize)
scatter = plt.scatter(
    reduced[:, 0], 
    reduced[:, 1], 
    c=labels, 
    cmap="tab10", 
    s=20, 
    alpha=0.8
)

plt.title(f"PCA Visualization with {n_clusters} Clusters")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="Cluster ID")
plt.tight_layout()
plt.show()

In [ ]:
for i in rep_indices:
    cur_rgb = all_rgbs[i]
    plt.imshow(cur_rgb)
    plt.show()